# D3/D4 Hands-on: CNN Classification

2026 Winter

本 Notebook 會自動設定環境、安裝依賴、並執行完整的訓練流程。

支援環境：**Google Colab** / **Kaggle** / **本機**

所有行為由 config YAML + dataset_info.yaml 驅動，無硬編碼的任務名稱。

In [ ]:
# Step 0: Select config
CONFIG_FILE = "config.yaml"              #@param ["config.yaml", "config_d4_hackathon.yaml"]

print(f"Using config: {CONFIG_FILE}")

In [ ]:
# Step 1: Detect environment
import sys, os
try:
    import google.colab
    ENV = "colab"
except ImportError:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        ENV = "kaggle"
    else:
        ENV = "local"

print(f"Detected environment: {ENV}")

In [ ]:
# Step 2: Mount Drive (Colab only)
if ENV == "colab":
    from google.colab import drive
    drive.mount("drive", force_remount=True)
    print("Drive mounted.")
else:
    print("Drive mount skipped (not Colab).")

In [ ]:
# Step 3: Clone repo
REPO_URL = "https://github.com/cbe135/d3-hands-on-liver-classification.git"
REPO_NAME = "d3-hands-on-liver-classification"

if os.path.exists(REPO_NAME):
    print(f"Repo already exists at ./{REPO_NAME}")
else:
    !git clone $REPO_URL

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Step 4: Install dependencies
if ENV == "local":
    !uv sync
else:
    !pip install -r requirements.txt
print("Dependencies installed.")

In [ ]:
# Step 5: Import libraries
import logging
import os
import yaml
from datetime import datetime
from tqdm.notebook import tqdm

import monai
import numpy as np
import torch
from matplotlib import pyplot as plt
from monai.utils.misc import set_determinism

print(f"MONAI: {monai.__version__}, PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
# Step 6: Load config (data-driven)
from src.main import load_config

args = load_config(CONFIG_FILE)

logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s.%(msecs)03d][%(levelname)5s](%(name)s) - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)
logger = logging.getLogger()
set_determinism(args["environ"]["seed"])
print(f"Data: {args['environ']['data_name']}")

In [ ]:
# Step 7: Setup data
from src.env_setup import setup_data, get_data_count, find_data_dir

setup_data(args)
get_data_count(args)
data_dir = find_data_dir(args)
print(f"Data directory: {data_dir}")

In [ ]:
# Step 8: Load data list + dataset_info (both data-driven)
from src.transforms import load_dataset_info, derive_has_masks, derive_reader

with open(os.path.join(data_dir, "data_list.yaml")) as f:
    data_dicts = yaml.safe_load(f)["data"]

dataset_info = load_dataset_info(data_dir)
print(f"Modality: {dataset_info.get('modality', 'unknown')}")
print(f"Has masks: {derive_has_masks(data_dicts)}")
print(f"Reader: {derive_reader(data_dicts) or 'monai default'}")

In [ ]:
# Step 9: Split data
from src.data import populate_data_lists

train_dicts, val_dicts, test_dicts = populate_data_lists(args, data_dicts)
print(f"{len(train_dicts)} training, {len(val_dicts)} validation, {len(test_dicts)} testing")

In [ ]:
# Step 10: Build transforms + datasets (derived from data)
from src.transforms import build_train_transform, build_val_transform
from src.data import generate_dataset, generate_dataloader

train_transform = build_train_transform(args, data_dicts, dataset_info)
val_transform = build_val_transform(args, data_dicts, dataset_info)

train_set = generate_dataset(args, train_dicts, train_transform)
val_set = generate_dataset(args, val_dicts, val_transform)
test_set = generate_dataset(args, test_dicts, val_transform)
print("Datasets ready.")

In [ ]:
# Step 11: Train
from src.train import train_pipeline
from src.utils import plot_loss_curves

print("Starting training...")
model, train_loader, val_loader, record = train_pipeline(args, train_set, val_set)
plot_loss_curves(args, record)
print("Training complete!")

In [ ]:
# Step 12: Evaluate
from src.evaluate import infer, plot_roc_and_show_result

best_state = torch.load("best_weights.pth", weights_only=True)
model.load_state_dict(best_state)

train_true, train_pred = infer(args, model, train_loader, True)
val_true, val_pred = infer(args, model, val_loader, True)
test_loader = generate_dataloader(args, test_set)
test_true, test_pred = infer(args, model, test_loader, True)

plot_roc_and_show_result(args, train_true, train_pred, title="Train")
plot_roc_and_show_result(args, val_true, val_pred, title="Validation")
plot_roc_and_show_result(args, test_true, test_pred, title="Test")

print("Evaluation complete!")